# KPIS

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, hour

spark = SparkSession.builder.appName("kpis_retail").getOrCreate()

project_id = "shaped-icon-478404-p0"
dataset_id = "retail_transaction"
temp_bucket = "retail-transactions-final"


# 1. Ticket Promedio
fact_df = spark.read \
    .format("bigquery") \
    .option("table", f"{project_id}:{dataset_id}.fact_transacciones") \
    .option("temporaryGcsBucket", temp_bucket) \
    .load()

kpi_ticket = fact_df.select(avg("Total_Cost").alias("ticket_promedio"))

# Guardar en GCS
kpi_ticket.write.mode("overwrite").parquet("gs://retail-transactions-final/kpis/ticket_promedio")


#  2. Horas Pico
from pyspark.sql.functions import hour

kpi_horas = fact_df.withColumn("hora", hour("Date")) \
    .groupBy("hora") \
    .agg(count("*").alias("cantidad_transacciones")) \
    .orderBy("cantidad_transacciones", ascending=False)

kpi_horas.write.mode("overwrite").parquet("gs://retail-transactions-final/kpis/horas_pico")


# 3. Frecuencia de compra
kpi_frecuencia = fact_df.groupBy("Customer_Name") \
    .agg(count("*").alias("frecuencia_compras")) \
    .orderBy("frecuencia_compras", ascending=False)

kpi_frecuencia.write.mode("overwrite").parquet("gs://retail-transactions-final/kpis/frecuencia_compras")


#  4. Promociones activas
kpi_promos = fact_df.groupBy("Promotion") \
    .agg(count("*").alias("transacciones")) \
    .orderBy("transacciones", ascending=False)

kpi_promos.write.mode("overwrite").parquet("gs://retail-transactions-final/kpis/promociones")

print("✅ KPIs generados y guardados en la carpeta /kpis/")


25/12/16 02:49:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


✅ KPIs generados y guardados en la carpeta /kpis/
